# Bitki Türü Tanıma Modeli — Eğitim Pipeline'ı
## Ne yapıyoruz?
- **93 bitki türü** için EfficientNetB3 modeli eğitiyoruz
- **PlantNet-300K** (200 görüntü/tür, mevcut) + **iNaturalist** (300 görüntü/tür, yeni indiriliyor) birleştiriyoruz
- iNaturalist: farklı açı, ışık, arka plan → model gerçek dünya fotoğraflarını tanıyacak
- **Agresif augmentation** ile veri çeşitliliği artırılıyor

## Veri klasörü (`data/`)
- **PlantNet-300K** (Kaggle): `data/plantnet_300K/` altında `images_train`, `images_val`, `images_test` ve her sınıfta sayısal tür klasörü (ör. `1355868/`). Küçük harf `plantnet_300k` da kabul edilir.
- **iNaturalist** indirmeleri: `data/inat_images/<bilimsel_ad_underscore>/`
- Birleştirme çıktısı: `data/combined_split_v2/{train,val,test}/plantnet__<id>/`

## Hücre sırası
1. Imports + sabitler
2. Tür listesi (93 sınıf)
3. iNaturalist görüntü indirme *(uzun sürebilir)*
4. Veri birleştirme (PlantNet + iNat → combined_split_v2)
5. tf.data pipeline + augmentation
6. Sınıf ağırlıkları
7. Model mimarisi (EfficientNetB3)
8. Aşama 1: Frozen backbone eğitimi
9. Aşama 2: Fine-tune
10. Değerlendirme + sınıf bazlı accuracy
11. Model + class_names kaydet
12. TFLite dönüştürme
13. TTA testi


In [1]:
# === 1. Imports + sabitler ===
import os, json, time, random, shutil
from pathlib import Path
import numpy as np
import requests
import warnings

warnings.filterwarnings("ignore")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.efficientnet import preprocess_input

# === Yol tanimlari (notebook nereden calisirsa calissin) ===
_cwd = Path.cwd().resolve()
if (_cwd / "requirements.txt").exists():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / "requirements.txt").exists():
    PROJECT_ROOT = _cwd.parent
elif (_cwd / "data").exists():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / "data").exists():
    PROJECT_ROOT = _cwd.parent
else:
    PROJECT_ROOT = _cwd

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
NAMES_DIR = PROJECT_ROOT / "class_names"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
for d in [MODELS_DIR, NAMES_DIR, OUTPUTS_DIR]:
    d.mkdir(exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

# === Sabitler ===
# EfficientNetB3 ImageNet girdisi 300x300; backbone ile hizali tutulur
IMG_SIZE = (300, 300)
BATCH_SIZE = 32
SEED = 42
N_INAT = 300  # Her tur icin iNaturalist hedef goruntu sayisi
AUTOTUNE = tf.data.AUTOTUNE

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


PROJECT_ROOT: C:\Users\HP\Desktop\plant_project_tur\plant_project
TensorFlow: 2.21.0
GPU: []


In [2]:
# === 2. Tur listesi (93 sinif) ===
# PlantNet species ID -> bilimsel ad (klasor adi: plantnet__<id>)
SPECIES = {
    "1355868": "Rosa canina",
    "1355937": "Quercus pubescens",
    "1355978": "Alnus glutinosa",
    "1355990": "Carpinus betulus",
    "1356022": "Populus tremula",
    "1356075": "Fraxinus excelsior",
    "1356126": "Acer pseudoplatanus",
    "1356257": "Tilia platyphyllos",
    "1356382": "Prunus spinosa",
    "1356420": "Malus sylvestris",
    "1356421": "Pyrus communis",
    "1356428": "Sorbus aucuparia",
    "1356692": "Robinia pseudoacacia",
    "1356781": "Gleditsia triacanthos",
    "1356816": "Cercis siliquastrum",
    "1357330": "Pinus sylvestris",
    "1357379": "Pinus pinea",
    "1357677": "Larix decidua",
    "1357705": "Pseudotsuga menziesii",
    "1358094": "Ilex aquifolium",
    "1358133": "Juglans regia",
    "1358150": "Paulownia tomentosa",
    "1358605": "Ficus carica",
    "1358689": "Morus alba",
    "1358751": "Betula pubescens",
    "1358766": "Corylus maxima",
    "1359197": "Galega officinalis",
    "1359483": "Prunus domestica",
    "1359485": "Prunus padus",
    "1359498": "Sorbus torminalis",
    "1359517": "Trifolium pratense",
    "1359525": "Fragaria vesca",
    "1359616": "Cytisus scoparius",
    "1359620": "Ulex europaeus",
    "1359622": "Genista tinctoria",
    "1359669": "Wisteria sinensis",
    "1360153": "Rhamnus cathartica",
    "1360588": "Koelreuteria paniculata",
    "1360590": "Rhododendron ponticum",
    "1360671": "Arbutus unedo",
    "1360811": "Ligustrum lucidum",
    "1360978": "Buddleja davidii",
    "1360998": "Callistemon citrinus",
    "1361024": "Myrtus communis",
    "1361656": "Juniperus horizontalis",
    "1361666": "Thuja occidentalis",
    "1361759": "Magnolia grandiflora",
    "1361823": "Magnolia kobus",
    "1361847": "Liriodendron tulipifera",
    "1361850": "Liquidambar styraciflua",
    "1362294": "Camellia sinensis",
    "1362490": "Philadelphus coronarius",
    "1362927": "Berberis thunbergii",
    "1362954": "Clematis montana",
    "1363021": "Pistacia lentiscus",
    "1363110": "Ceanothus thyrsiflorus",
    "1363126": "Hibiscus syriacus",
    "1363128": "Lavandula angustifolia",
    "1363130": "Rosmarinus officinalis",
    "1363227": "Nerium oleander",
    "1363336": "Jasminum officinale",
    "1363451": "Phyllostachys aurea",
    "1363490": "Agave americana",
    "1363699": "Eucalyptus globulus",
    "1363737": "Trifolium dubium",
    "1363740": "Trifolium repens",
    "1363749": "Caragana arborescens",
    "1363764": "Styphnolobium japonicum",
    "1363778": "Acacia retinodes",
    "1363871": "Barbarea verna",
    "1364099": "Centranthus ruber",
    "1367432": "Lupinus polyphyllus",
    "1369887": "Trachelospermum jasminoides",
    "1369960": "Tradescantia spathacea",
    "1372016": "Morinda citrifolia",
    "1374048": "Tagetes erecta",
    "1385937": "Zamioculcas zamiifolia",
    "1389231": "Phedimus spurius",
    "1389297": "Cereus jamacaru",
    "1389307": "Sedum pachyphyllum",
    "1391112": "Chaerophyllum hirsutum",
    "1391192": "Cirsium eriophorum",
    "1391226": "Cirsium oleraceum",
    "1391483": "Cucurbita maxima",
    "1391652": "Daphne mezereum",
    "1391797": "Dryas octopetala",
    "1391953": "Epipactis atrorubens",
    "1391963": "Epipactis palustris",
    "1392094": "Erucastrum incanum",
    "1392654": "Gomphocarpus physocarpus",
    "1392695": "Hebe salicifolia",
    "1392777": "Hippophae rhamnoides",
    "1393241": "Hypericum calycinum",
}

CLASS_NAMES = sorted(f"plantnet__{pid}" for pid in SPECIES)
NUM_CLASSES = len(CLASS_NAMES)
CLASS_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}
print(f"Toplam sinif: {NUM_CLASSES}")


Toplam sinif: 93


In [3]:
# === 3. iNaturalist goruntu indirme (uzun surebilir) ===
# Research-grade gozlemler; mevcut dosyalar atlanir (yeniden calistirmada devam)

INAT_DIR = DATA_DIR / "inat_images"
INAT_DIR.mkdir(exist_ok=True)

def get_taxon_id(scientific_name):
    r = requests.get(
        "https://api.inaturalist.org/v1/taxa",
        params={"q": scientific_name, "per_page": 5},
        headers={"Accept": "application/json"},
        timeout=15,
    )
    for res in r.json().get("results", []):
        if res.get("name", "").lower() == scientific_name.lower() and res.get("rank") == "species":
            return res["id"]
    results = r.json().get("results", [])
    for res in results:
        if res.get("rank") == "species":
            return res["id"]
    return None

def download_species(scientific_name, n_target=N_INAT):
    folder = INAT_DIR / scientific_name.replace(" ", "_")
    folder.mkdir(exist_ok=True)
    existing = {p.stem for p in folder.glob("*.jpg")}
    if len(existing) >= n_target:
        return len(existing)

    taxon_id = get_taxon_id(scientific_name)
    if not taxon_id:
        print(f"  UYARI: taxon bulunamadi -> {scientific_name}")
        return 0

    downloaded = len(existing)
    for page in range(1, 8):
        if downloaded >= n_target:
            break
        try:
            r = requests.get(
                "https://api.inaturalist.org/v1/observations",
                params={
                    "taxon_id": taxon_id,
                    "quality_grade": "research",
                    "photos": "true",
                    "per_page": 200,
                    "page": page,
                    "order": "random",
                },
                headers={"Accept": "application/json"},
                timeout=30,
            )
            obs_list = r.json().get("results", [])
        except Exception as e:
            print(f"  Hata (sayfa {page}): {e}")
            break
        if not obs_list:
            break

        for obs in obs_list:
            if downloaded >= n_target:
                break
            photos = obs.get("photos", [])
            if not photos:
                continue
            pid = str(photos[0].get("id", ""))
            if pid in existing:
                continue
            url = photos[0].get("url", "").replace("/square.", "/medium.")
            if not url.startswith("http"):
                continue
            try:
                img_r = requests.get(url, timeout=20)
                if img_r.status_code == 200 and len(img_r.content) > 5000:
                    (folder / f"{pid}.jpg").write_bytes(img_r.content)
                    existing.add(pid)
                    downloaded += 1
            except:
                pass
            time.sleep(0.12)

        time.sleep(0.5)

    return downloaded

# Tum turleri indir
print(f"{NUM_CLASSES} tur icin iNaturalist indirme basliyor...")
print("Her tur ~300 goruntu. Kesintiye ugrarsa yeniden calistir, devam eder.\n")

for i, (pid, sci_name) in enumerate(SPECIES.items(), 1):
    n = download_species(sci_name)
    print(f"[{i:>3}/{NUM_CLASSES}] {sci_name:<40} {n} goruntu")
    time.sleep(0.3)

print("\nIndirme tamamlandi!")


93 tur icin iNaturalist indirme basliyor...
Her tur ~300 goruntu. Kesintiye ugrarsa yeniden calistir, devam eder.

[  1/93] Rosa canina                              300 goruntu
[  2/93] Quercus pubescens                        300 goruntu
[  3/93] Alnus glutinosa                          300 goruntu
[  4/93] Carpinus betulus                         300 goruntu
[  5/93] Populus tremula                          300 goruntu
[  6/93] Fraxinus excelsior                       300 goruntu
[  7/93] Acer pseudoplatanus                      300 goruntu
[  8/93] Tilia platyphyllos                       300 goruntu
[  9/93] Prunus spinosa                           300 goruntu
[ 10/93] Malus sylvestris                         300 goruntu
[ 11/93] Pyrus communis                           300 goruntu
[ 12/93] Sorbus aucuparia                         300 goruntu
[ 13/93] Robinia pseudoacacia                     300 goruntu
[ 14/93] Gleditsia triacanthos                    300 goruntu
[ 15/93] Cercis s

In [4]:
# === 4. Veri birlestirme (PlantNet + iNat -> combined_split_v2) ===
# PlantNet kaynak (otomatik):
#   - Kaggle ciktisi: data/plantnet_300K/images_{train,val,test}/<tur_id>/*.jpg
#     (klasor adi buyuk K: plantnet_300K; kucuk harfli plantnet_300k de kabul)
#   - Eski yapi: data/combined_split/{train,val,test}/plantnet__<tur_id>/
# iNat: 240 train / 30 val / 30 test (toplam 300/sinif hedef)

PLANTNET_RAW = DATA_DIR / "plantnet_300K"
if not PLANTNET_RAW.is_dir():
    _alt_raw = DATA_DIR / "plantnet_300k"
    if _alt_raw.is_dir():
        PLANTNET_RAW = _alt_raw

PLANTNET_SPLIT = DATA_DIR / "combined_split"
PLANTNET_RAW_SPLITS = {
    "train": PLANTNET_RAW / "images_train",
    "val": PLANTNET_RAW / "images_val",
    "test": PLANTNET_RAW / "images_test",
}
USE_KAGGLE_PLANTNET = (PLANTNET_RAW_SPLITS["train"]).is_dir()

OUT_SPLIT = DATA_DIR / "combined_split_v2"
INAT_TRAIN, INAT_VAL, INAT_TEST = 240, 30, 30

if USE_KAGGLE_PLANTNET:
    print("PlantNet kaynak:", PLANTNET_RAW, "(Kaggle images_train/val/test)")
else:
    print("PlantNet kaynak:", PLANTNET_SPLIT, "(combined_split)")

for split in ["train", "val", "test"]:
    (OUT_SPLIT / split).mkdir(parents=True, exist_ok=True)


def _copy_images(src_dir: Path, dst_dir: Path) -> None:
    if not src_dir.is_dir():
        return
    for img in sorted(src_dir.iterdir()):
        if img.is_file() and img.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
            shutil.copy2(img, dst_dir / img.name)


for cls_name in CLASS_NAMES:
    pid = cls_name.replace("plantnet__", "")
    sci_name = SPECIES[pid]

    # PlantNet goruntuleri kopyala
    for split in ["train", "val", "test"]:
        dst = OUT_SPLIT / split / cls_name
        dst.mkdir(exist_ok=True)
        if USE_KAGGLE_PLANTNET:
            src = PLANTNET_RAW_SPLITS[split] / pid
        else:
            src = PLANTNET_SPLIT / split / cls_name
        _copy_images(src, dst)

    # iNat goruntuleri ekle
    inat_dir = INAT_DIR / sci_name.replace(" ", "_")
    if inat_dir.exists():
        imgs = list(inat_dir.glob("*.jpg"))
        random.shuffle(imgs)
        splits_inat = [
            ("train", imgs[:INAT_TRAIN]),
            ("val",   imgs[INAT_TRAIN:INAT_TRAIN + INAT_VAL]),
            ("test",  imgs[INAT_TRAIN + INAT_VAL:INAT_TRAIN + INAT_VAL + INAT_TEST]),
        ]
        for split_name, split_imgs in splits_inat:
            dst = OUT_SPLIT / split_name / cls_name
            dst.mkdir(exist_ok=True)
            for img in split_imgs:
                shutil.copy2(img, dst / img.name)

# Sayim (jpg/jpeg/png/webp)
_img_suffixes = {".jpg", ".jpeg", ".png", ".webp"}


def _count_class_images(split: str, cls: str) -> int:
    p = OUT_SPLIT / split / cls
    if not p.is_dir():
        return 0
    return sum(1 for f in p.iterdir() if f.is_file() and f.suffix.lower() in _img_suffixes)


for split in ["train", "val", "test"]:
    total = sum(_count_class_images(split, c) for c in CLASS_NAMES)
    print(f"{split:5s}: {total:>6} goruntu ({total // NUM_CLASSES:.0f} ort./sinif)")


PlantNet kaynak: C:\Users\HP\Desktop\plant_project_tur\plant_project\data\plantnet_300K (Kaggle images_train/val/test)
train: 138197 goruntu (1485 ort./sinif)
val  :  19166 goruntu (206 ort./sinif)
test :  19162 goruntu (206 ort./sinif)


In [5]:
# === 5. tf.data pipeline + augmentation ===
# Augment: [0,255] float uzayinda; ardindan EfficientNet preprocess_input

TRAIN_DIR = str(OUT_SPLIT / "train")
VAL_DIR = str(OUT_SPLIT / "val")
TEST_DIR = str(OUT_SPLIT / "test")

aug_layer = keras.Sequential(
    [
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.30),
        layers.RandomZoom(0.25),
        layers.RandomContrast(0.30),
        layers.RandomBrightness(0.25),
        layers.RandomTranslation(0.10, 0.10),
    ],
    name="augmentation",
)


def make_dataset(directory, shuffle=False, augment=False):
    ds = keras.utils.image_dataset_from_directory(
        directory,
        labels="inferred",
        label_mode="int",
        class_names=CLASS_NAMES,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED,
    )

    def train_step(img, label):
        x = tf.cast(img, tf.float32)
        x = aug_layer(x, training=True)
        x = preprocess_input(x)
        return x, label

    def eval_step(img, label):
        x = tf.cast(img, tf.float32)
        x = preprocess_input(x)
        return x, label

    if augment:
        ds = ds.map(train_step, num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(eval_step, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_ds = make_dataset(TRAIN_DIR, shuffle=True, augment=True)
val_ds = make_dataset(VAL_DIR, shuffle=False, augment=False)
test_ds = make_dataset(TEST_DIR, shuffle=False, augment=False)

print("Train:", int(train_ds.cardinality().numpy()), "batch")
print("Val:  ", int(val_ds.cardinality().numpy()), "batch")
print("Test: ", int(test_ds.cardinality().numpy()), "batch")


Found 138197 files belonging to 93 classes.
Found 19166 files belonging to 93 classes.
Found 19162 files belonging to 93 classes.
Train: 4319 batch
Val:   599 batch
Test:  599 batch


In [24]:
# === 6. Sinif agirliklari ===
from sklearn.utils.class_weight import compute_class_weight

all_labels = []
for _, labels in train_ds.unbatch():
    all_labels.append(int(labels.numpy()))
all_labels = np.array(all_labels)

weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=all_labels)
class_weight_dict = {i: float(w) for i, w in enumerate(weights)}

print(f"Agirlik araliği: {min(weights):.3f} - {max(weights):.3f}")

Agirlik araliği: 0.191 - 5.231


In [25]:
# === 7. Model mimarisi (EfficientNetB3) ===
# ImageNet pretrained backbone -> GAP -> Dropout(0.4) -> Dense(softmax)

base_model = keras.applications.EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=IMG_SIZE + (3,),
)
base_model.trainable = False  # Asama 1'de dondurulmus

inputs  = keras.Input(shape=IMG_SIZE + (3,))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dropout(0.4)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax",
                       kernel_regularizer=keras.regularizers.l2(1e-4))(x)

model = keras.Model(inputs, outputs)
model.summary(line_length=80, show_trainable=True)


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape           ┃     Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer_5 (InputLayer)    │ (None, 300, 300, 3)    │           0 │   -   │
├───────────────────────────────┼────────────────────────┼─────────────┼───────┤
│ efficientnetb3 (Functional)   │ (None, 10, 10, 1536)   │  10,783,535 │   N   │
├───────────────────────────────┼────────────────────────┼─────────────┼───────┤
│ global_average_pooling2d_1    │ (None, 1536)           │           0 │   -   │
│ (GlobalAveragePooling2D)      │                        │             │       │
├───────────────────────────────┼────────────────────────┼─────────────┼───────┤
│ dropout_1 (Dropout)           │ (None, 1536)           │           0 │   -   │
├───────────────────────────────┼────────────────────────┼─────────────┼───────┤
│ dense_1 (Dense)               │ (None, 93)             │     142,941 │   Y   │
└───────────────────────────────┴────────────────────────┴─────────────┴───────┘

 Total params: 10,926,476 (41.68 MB)

 Trainable params: 142,941 (558.36 KB)

 Non-trainable params: 10,783,535 (41.14 MB)

In [8]:
# === 8. Asama 1: Frozen backbone egitimi ===

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

cb_checkpoint = keras.callbacks.ModelCheckpoint(
    str(MODELS_DIR / "efficientnetb3_stage1.keras"),
    save_best_only=True, monitor="val_accuracy", verbose=1,
)
cb_earlystop = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=4, restore_best_weights=True, verbose=1,
)

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[cb_checkpoint, cb_earlystop],
    class_weight=class_weight_dict,
)

print("\nAsama 1 tamamlandi.")
print(f"En iyi val_acc: {max(history1.history['val_accuracy']):.4f}")


Epoch 1/10
4186/4186 ━━━━━━━━━━━━━━━━━━━━ 0s 629ms/step - accuracy: 0.3756 - loss: 3.0679
Epoch 1: val_accuracy improved from -inf to 0.60494, saving model to /Users/erendeveci/Downloads/plant-project/models/efficientnetb3_stage1.keras
4186/4186 ━━━━━━━━━━━━━━━━━━━━ 2953s 704ms/step - accuracy: 0.3757 - loss: 3.0678 - val_accuracy: 0.6049 - val_loss: 1.7001
Epoch 2/10
4186/4186 ━━━━━━━━━━━━━━━━━━━━ 0s 631ms/step - accuracy: 0.4915 - loss: 2.3753
Epoch 2: val_accuracy improved from 0.60494 to 0.62278, saving model to /Users/erendeveci/Downloads/plant-project/models/efficientnetb3_stage1.keras
4186/4186 ━━━━━━━━━━━━━━━━━━━━ 2956s 706ms/step - accuracy: 0.4915 - loss: 2.3753 - val_accuracy: 0.6228 - val_loss: 1.6461
Epoch 3/10
4186/4186 ━━━━━━━━━━━━━━━━━━━━ 0s 632ms/step - accuracy: 0.4995 - loss: 2.3373
Epoch 3: val_accuracy improved from 0.62278 to 0.63018, saving model to /Users/erendeveci/Downloads/plant-project/models/efficientnetb3_stage1.keras
4186/4186 ━━━━━━━━━━━━━━━━━━━━ 2960s 7

In [9]:
# === 9. Asama 2: Fine-tune ===

base_model.trainable = True
fine_tune_at = len(base_model.layers) - 40
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

trainable_count = sum(1 for l in model.layers if l.trainable)
print(f"Egitilen katman: {trainable_count}")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

cb_checkpoint2 = keras.callbacks.ModelCheckpoint(
    str(MODELS_DIR / "efficientnetb3_93classes.keras"),
    save_best_only=True, monitor="val_accuracy", verbose=1,
)
cb_earlystop2 = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=6, restore_best_weights=True, verbose=1,
)
cb_reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1,
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    callbacks=[cb_checkpoint2, cb_earlystop2, cb_reduce_lr],
    class_weight=class_weight_dict,
)

print("\nAsama 2 tamamlandi.")
print(f"En iyi val_acc: {max(history2.history['val_accuracy']):.4f}")


Egitilen katman: 5
Epoch 1/25
4186/4186 ━━━━━━━━━━━━━━━━━━━━ 0s 799ms/step - accuracy: 0.4037 - loss: 2.7598
Epoch 1: val_accuracy improved from -inf to 0.65495, saving model to /Users/erendeveci/Downloads/plant-project/models/efficientnetb3_93classes.keras
4186/4186 ━━━━━━━━━━━━━━━━━━━━ 3663s 873ms/step - accuracy: 0.4037 - loss: 2.7597 - val_accuracy: 0.6550 - val_loss: 1.5854 - learning_rate: 1.0000e-05
Epoch 2/25
4186/4186 ━━━━━━━━━━━━━━━━━━━━ 0s 798ms/step - accuracy: 0.5584 - loss: 2.0523
Epoch 2: val_accuracy improved from 0.65495 to 0.68861, saving model to /Users/erendeveci/Downloads/plant-project/models/efficientnetb3_93classes.keras
4186/4186 ━━━━━━━━━━━━━━━━━━━━ 3652s 872ms/step - accuracy: 0.5584 - loss: 2.0523 - val_accuracy: 0.6886 - val_loss: 1.4334 - learning_rate: 1.0000e-05
Epoch 3/25
4186/4186 ━━━━━━━━━━━━━━━━━━━━ 0s 798ms/step - accuracy: 0.5974 - loss: 1.8515
Epoch 3: val_accuracy improved from 0.68861 to 0.70866, saving model to /Users/erendeveci/Downloads/plant-

In [10]:
# === 10. Degerlendirme + sinif bazli accuracy ===
import pandas as pd

# Test seti genel accuracy
loss, acc = model.evaluate(test_ds, verbose=0)
print(f"Test Loss    : {loss:.4f}")
print(f"Test Accuracy: {acc:.4f}  ({acc*100:.1f}%)")

# Sinif bazli accuracy
y_true, y_pred = [], []
for imgs, labels in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

per_class = []
for i, cls_name in enumerate(CLASS_NAMES):
    mask = y_true == i
    if mask.sum() == 0:
        continue
    cls_acc = (y_pred[mask] == i).mean() * 100
    pid = cls_name.replace("plantnet__", "")
    sci_name = SPECIES.get(pid, cls_name)
    per_class.append({"Tur": sci_name, "Accuracy": round(cls_acc, 1)})

df = pd.DataFrame(per_class).sort_values("Accuracy")
print(f"\n%70 alti siniflar: {(df['Accuracy'] < 70).sum()}")
print(f"%80 ve ustu siniflar: {(df['Accuracy'] >= 80).sum()}")
print("\n--- EN DUSUK 15 ---")
print(df.head(15).to_string(index=False))
print("\n--- EN YUKSEK 10 ---")
print(df.tail(10).to_string(index=False))


Test Loss    : 0.8627
Test Accuracy: 0.8183  (81.8%)

%70 alti siniflar: 12
%80 ve ustu siniflar: 57

--- EN DUSUK 15 ---
                    Tur  Accuracy
      Hibiscus syriacus      31.7
   Caragana arborescens      49.5
Liquidambar styraciflua      62.3
  Pseudotsuga menziesii      64.2
          Juglans regia      66.1
Philadelphus coronarius      66.3
    Eucalyptus globulus      68.0
          Larix decidua      68.6
         Pyrus communis      68.7
       Malus sylvestris      69.1
       Sorbus aucuparia      69.6
     Thuja occidentalis      69.7
     Erucastrum incanum      72.4
 Lavandula angustifolia      72.9
          Arbutus unedo      73.1

--- EN YUKSEK 10 ---
                   Tur  Accuracy
   Epipactis palustris      91.8
      Cucurbita maxima      92.3
      Dryas octopetala      93.1
   Hypericum calycinum      93.8
        Fragaria vesca      94.4
    Morinda citrifolia      94.8
       Cereus jamacaru      94.9
Zamioculcas zamiifolia      95.4
Tradescantia sp

2026-04-13 18:16:33.686876: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [11]:
# === 11. Model + class_names kaydet ===
model_path = MODELS_DIR / "efficientnetb3_93classes.keras"
model.save(str(model_path))
print("Model kaydedildi:", model_path)

names_path = NAMES_DIR / "class_names.json"
with open(names_path, "w", encoding="utf-8") as f:
    json.dump(CLASS_NAMES, f, ensure_ascii=False, indent=2)
print("Class names kaydedildi:", names_path)

id_map_path = NAMES_DIR / "plantnet_species_id_map.json"
with open(id_map_path, "w", encoding="utf-8") as f:
    json.dump(SPECIES, f, ensure_ascii=False, indent=2)
print("ID haritasi kaydedildi:", id_map_path)

hist_all = dict(history1.history)
if "history2" in globals():
    for k, v in history2.history.items():
        hist_all[k] = hist_all.get(k, []) + v
with open(OUTPUTS_DIR / "training_history_93classes.json", "w", encoding="utf-8") as f:
    json.dump(hist_all, f, indent=2)
print("Egitim gecmisi kaydedildi: training_history_93classes.json")


Model kaydedildi: /Users/erendeveci/Downloads/plant-project/models/efficientnetb3_93classes.keras
Class names kaydedildi: /Users/erendeveci/Downloads/plant-project/class_names/class_names.json
ID haritasi kaydedildi: /Users/erendeveci/Downloads/plant-project/class_names/plantnet_species_id_map.json
Egitim gecmisi kaydedildi: training_history_93classes.json


In [12]:
# === 12. TFLite donusturme ===
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = []
tflite_model = converter.convert()

tflite_path = MODELS_DIR / "plant_model_93classes.tflite"
tflite_path.write_bytes(tflite_model)
print(f"TFLite kaydedildi: {tflite_path}")
print(f"Boyut: {tflite_path.stat().st_size / 1024 / 1024:.1f} MB")

shutil.copy(str(NAMES_DIR / "class_names.json"), str(MODELS_DIR / "class_names.json"))
print("class_names.json modeller klasorune kopyalandi.")


INFO:tensorflow:Assets written to: /var/folders/p4/bvxky_m90y72dj4chpyk3w240000gn/T/tmp9940fo98/assets


INFO:tensorflow:Assets written to: /var/folders/p4/bvxky_m90y72dj4chpyk3w240000gn/T/tmp9940fo98/assets


Saved artifact at '/var/folders/p4/bvxky_m90y72dj4chpyk3w240000gn/T/tmp9940fo98'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 300, 300, 3), dtype=tf.float32, name='keras_tensor_392')
Output Type:
  TensorSpec(shape=(None, 93), dtype=tf.float32, name=None)
Captures:
  5258076224: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5258076752: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  5245918896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5232218352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5232218528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5232215312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5232217248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5232221872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5226480032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5226480208: TensorSpec(shape=(), dtype=tf.resource, name=None)

W0000 00:00:1776093404.233048   55380 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1776093404.233070   55380 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-04-13 18:16:44.234353: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/p4/bvxky_m90y72dj4chpyk3w240000gn/T/tmp9940fo98
2026-04-13 18:16:44.245126: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-04-13 18:16:44.245135: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/p4/bvxky_m90y72dj4chpyk3w240000gn/T/tmp9940fo98
I0000 00:00:1776093404.333359   55380 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-04-13 18:16:44.349244: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-04-13 18:16:44.878363: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /var/folder

TFLite kaydedildi: /Users/erendeveci/Downloads/plant-project/models/plant_model_93classes.tflite
Boyut: 41.3 MB
class_names.json modeller klasorune kopyalandi.


In [13]:
# === 13. TTA testi ===
# test_ds ciktisi zaten preprocess_input uygulanmis; burada tekrar preprocess yok


def predict_with_tta(model, img_batch, n_variants=8):
    variants = []
    for k in range(4):
        r = np.rot90(img_batch, k=k, axes=(1, 2))
        variants.append(np.array(r, copy=True))
        variants.append(np.array(r[:, :, ::-1, :], copy=True))
    preds = [model.predict(v, verbose=0) for v in variants[:n_variants]]
    return np.mean(preds, axis=0)

y_true_tta, y_pred_normal, y_pred_tta = [], [], []

for imgs, labels in test_ds:
    imgs_np = imgs.numpy()
    lbls_np = labels.numpy()
    p_normal = model.predict(imgs_np, verbose=0)
    p_tta    = predict_with_tta(model, imgs_np)
    y_true_tta.extend(lbls_np)
    y_pred_normal.extend(np.argmax(p_normal, axis=1))
    y_pred_tta.extend(np.argmax(p_tta,    axis=1))

y_true_tta    = np.array(y_true_tta)
y_pred_normal = np.array(y_pred_normal)
y_pred_tta    = np.array(y_pred_tta)

acc_normal = (y_pred_normal == y_true_tta).mean()
acc_tta    = (y_pred_tta    == y_true_tta).mean()

print(f"Normal Accuracy : {acc_normal:.4f}  ({acc_normal*100:.1f}%)")
print(f"TTA    Accuracy : {acc_tta:.4f}  ({acc_tta*100:.1f}%)")
print(f"TTA kazanimi    : +{(acc_tta - acc_normal)*100:.2f} puan")


Normal Accuracy : 0.8183  (81.8%)
TTA    Accuracy : 0.8314  (83.1%)
TTA kazanimi    : +1.31 puan


In [3]:
# === 14. Tum siniflar icin accuracy degerleri ===
y_true2, y_pred2 = [], []
for imgs, labels in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_true2.extend(labels.numpy())
    y_pred2.extend(np.argmax(preds, axis=1))

y_true2 = np.array(y_true2)
y_pred2 = np.array(y_pred2)

per_class_all = []
for i, cls_name in enumerate(CLASS_NAMES):
    mask = y_true2 == i
    if mask.sum() == 0:
        continue
    cls_acc = (y_pred2[mask] == i).mean() * 100
    pid = cls_name.replace("plantnet__", "").replace("leafsnap__", "")
    sci_name = SPECIES.get(pid, pid)
    per_class_all.append({"Sinif": cls_name, "Tur": sci_name, "Accuracy": round(cls_acc, 1)})

df_all = pd.DataFrame(per_class_all).sort_values("Accuracy", ascending=False).reset_index(drop=True)
df_all.index += 1
print(f"Toplam sinif sayisi: {len(df_all)}")
print(f"Genel dogruluk: %{(y_true2 == y_pred2).mean()*100:.1f}")
print()
print(df_all.to_string())


Toplam sinif sayisi: 93
Genel dogruluk: %80.1

                Sinif                          Tur  Accuracy
1   plantnet__1389307           Sedum pachyphyllum     100.0
2   plantnet__1369960       Tradescantia spathacea      97.1
3   plantnet__1385937       Zamioculcas zamiifolia      95.4
4   plantnet__1372016           Morinda citrifolia      95.3
5   plantnet__1391963          Epipactis palustris      94.4
6   plantnet__1359525               Fragaria vesca      93.9
7   plantnet__1389297              Cereus jamacaru      93.7
8   plantnet__1391797             Dryas octopetala      93.0
9   plantnet__1393241          Hypericum calycinum      93.0
10  plantnet__1363490              Agave americana      92.2
11  plantnet__1391483             Cucurbita maxima      91.2
12  plantnet__1391192           Cirsium eriophorum      90.3
13  plantnet__1363110       Ceanothus thyrsiflorus      89.9
14  plantnet__1359517           Trifolium pratense      89.9
15  plantnet__1363336          Jasminu

In [1]:
# === Model + test_ds yukle (diger hucreler calistirilmadiysa bunu once calistir) ===
import json
import numpy as np
import pandas as pd
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input

_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd if (_cwd / "requirements.txt").exists() else _cwd.parent
DATA_DIR   = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
NAMES_DIR  = PROJECT_ROOT / "class_names"

model = tf.keras.models.load_model(str(MODELS_DIR / "efficientnetb3_93classes.keras"))
print("Model yuklendi.")

with open(NAMES_DIR / "class_names.json", encoding="utf-8") as f:
    CLASS_NAMES = json.load(f)
NUM_CLASSES = len(CLASS_NAMES)

id_map_path = NAMES_DIR / "plantnet_species_id_map.json"
SPECIES = json.load(open(id_map_path, encoding="utf-8")) if id_map_path.exists() else {}

def _eval_step(img, label):
    return preprocess_input(tf.cast(img, tf.float32)), label

test_ds = (
    tf.keras.utils.image_dataset_from_directory(
        str(DATA_DIR / "combined_split_v2" / "test"),
        labels="inferred", label_mode="int",
        class_names=CLASS_NAMES,
        image_size=(300, 300), batch_size=32,
        shuffle=False, seed=42,
    )
    .map(_eval_step, num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)
print(f"Test dataseti hazir: {int(test_ds.cardinality().numpy())} batch")

Model yuklendi.
Found 19162 files belonging to 93 classes.
Test dataseti hazir: 599 batch


In [4]:
# === Sonuclari kaydet ===
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report

_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd if (_cwd / "requirements.txt").exists() else _cwd.parent
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

# Tahminleri hesapla
y_true_s, y_pred_s = [], []
for imgs, labels in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_true_s.extend(labels.numpy())
    y_pred_s.extend(np.argmax(preds, axis=1))
y_true_s = np.array(y_true_s)
y_pred_s = np.array(y_pred_s)

# df_all olustur
rows = []
for i, cls_name in enumerate(CLASS_NAMES):
    mask = y_true_s == i
    if mask.sum() == 0:
        continue
    pid = cls_name.replace("plantnet__", "").replace("leafsnap__", "")
    sci_name = SPECIES.get(pid, cls_name)
    rows.append({
        "Sinif": cls_name,
        "Tur": sci_name,
        "Accuracy (%)": round((y_pred_s[mask] == i).mean() * 100, 1),
        "Dogru": int((y_pred_s[mask] == i).sum()),
        "Toplam": int(mask.sum()),
    })
df_all = pd.DataFrame(rows).sort_values("Accuracy (%)", ascending=False).reset_index(drop=True)
df_all.index += 1

# CSV kaydet
csv_path = OUTPUTS_DIR / "per_class_accuracy_93classes.csv"
df_all.to_csv(csv_path, index=True, encoding="utf-8-sig")
print("CSV kaydedildi:", csv_path)

# JSON kaydet
json_path = OUTPUTS_DIR / "per_class_accuracy_93classes.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(df_all.to_dict(orient="records"), f, ensure_ascii=False, indent=2)
print("JSON kaydedildi:", json_path)
print(f"Toplam {len(df_all)} sinif kaydedildi.")

# Classification report
clf_report = classification_report(y_true_s, y_pred_s, labels=range(NUM_CLASSES),
                                   target_names=CLASS_NAMES, output_dict=True)
clf_report_path = OUTPUTS_DIR / "classification_report_93classes.json"
with open(clf_report_path, "w", encoding="utf-8") as f:
    json.dump(clf_report, f, ensure_ascii=False, indent=2)
print(f"Classification report kaydedildi: {clf_report_path}")

# Genel metrikler
metrics = {
    "Overall Accuracy": round((y_true_s == y_pred_s).mean() * 100, 2),
    "Total Samples": int(len(y_true_s)),
    "Correct Predictions": int((y_true_s == y_pred_s).sum()),
    "Macro Avg F1-Score": round(clf_report["macro avg"]["f1-score"], 4),
    "Weighted Avg F1-Score": round(clf_report["weighted avg"]["f1-score"], 4),
}
metrics_path = OUTPUTS_DIR / "metrics_93classes.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print(f"Metrikler kaydedildi: {metrics_path}")
print(f"\nGenel Dogruluk : %{metrics['Overall Accuracy']}")
print(f"F1 (Weighted)  : {metrics['Weighted Avg F1-Score']}")

CSV kaydedildi: C:\Users\HP\Desktop\plant_project_tur\plant_project\outputs\per_class_accuracy_93classes.csv
JSON kaydedildi: C:\Users\HP\Desktop\plant_project_tur\plant_project\outputs\per_class_accuracy_93classes.json
Toplam 93 sinif kaydedildi.
Classification report kaydedildi: C:\Users\HP\Desktop\plant_project_tur\plant_project\outputs\classification_report_93classes.json
Metrikler kaydedildi: C:\Users\HP\Desktop\plant_project_tur\plant_project\outputs\metrics_93classes.json

Genel Dogruluk : %80.11
F1 (Weighted)  : 0.8069


In [ ]:
# === TFLite Dönüştürme ===
import shutil

tflite_out = MODELS_DIR / "plant_model_93classes.tflite"

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # float16 / dynamic range quantization
tflite_model = converter.convert()

tflite_out.write_bytes(tflite_model)
print(f"TFLite kaydedildi : {tflite_out}")
print(f"Boyut             : {tflite_out.stat().st_size / 1024 / 1024:.1f} MB")

# class_names.json'u models/ klasörüne de kopyala (mobil uygulama için)
names_src = NAMES_DIR / "class_names.json"
names_dst = MODELS_DIR / "class_names.json"
shutil.copy(str(names_src), str(names_dst))
print(f"class_names.json kopyalandı: {names_dst}")